# Loghub - HDFS

## Phase 1: Parse Log với Drain3

In [8]:
import pandas as pd

In [23]:
path_template = r'..\..\datasets\loghub\HDFS\HDFS_templates.csv'
path_log_structured = r'..\..\datasets\loghub\HDFS\HDFS_2k.log_structured.csv'
path_log_template = r'..\..\datasets\loghub\HDFS\HDFS_2k.log_templates.csv'

df_template = pd.read_csv(path_template)
df_log_structured = pd.read_csv(path_log_structured)
df_log_template = pd.read_csv(path_log_template)

print(f'Size df_template: {len(df_template)}')
print(f'Size df_log_structured: {len(df_log_structured)}')
print(f'Size df_log_template: {len(df_log_template)}')
df_log_structured.head()

Size df_template: 30
Size df_log_structured: 2000
Size df_log_template: 14


,LineId,Date,Time,Pid,Level,Component,Content,EventId,EventTemplate
0,1,81109,203615,148,INFO,dfs.DataNode$PacketResponder,PacketResponder 1 for block blk_38865049064139...,E10,PacketResponder <*> for block blk_<*> terminating
1,2,81109,203807,222,INFO,dfs.DataNode$PacketResponder,PacketResponder 0 for block blk_-6952295868487...,E10,PacketResponder <*> for block blk_<*> terminating
2,3,81109,204005,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.addStoredBlock: blockMap upd...,E6,BLOCK* NameSystem.addStoredBlock: blockMap upd...
3,4,81109,204015,308,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_82291938032499...,E10,PacketResponder <*> for block blk_<*> terminating
4,5,81109,204106,329,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_-6670958622368...,E10,PacketResponder <*> for block blk_<*> terminating


log file có 2000 dòng

In [24]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

config = TemplateMinerConfig()
config.drain_sim_th = 0.4        
config.drain_depth = 4        

miner = TemplateMiner(config=config)
log_lines = df_log_structured['Content'].astype(str).tolist()
print(f"Processing {len(log_lines)} log lines...")

for line in log_lines:
    result = miner.add_log_message(line)
    print(f"Cluster ID: {result['cluster_id']}")
    print(f"Template:   {result['template_mined']}")
    print(f"Change:     {result['change_type']}")
    print()


Processing 2000 log lines...
Cluster ID: 1
Template:   PacketResponder 1 for block blk_38865049064139660 terminating
Change:     cluster_created

Cluster ID: 1
Template:   PacketResponder <*> for block <*> terminating
Change:     cluster_template_changed

Cluster ID: 2
Template:   BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
Change:     cluster_created

Cluster ID: 1
Template:   PacketResponder <*> for block <*> terminating
Change:     none

Cluster ID: 1
Template:   PacketResponder <*> for block <*> terminating
Change:     none

Cluster ID: 2
Template:   BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size 67108864
Change:     cluster_template_changed

Cluster ID: 2
Template:   BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size 67108864
Change:     none

Cluster ID: 2
Template:   BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size 6710

In [35]:
print("=== All Templates ===")
template_data = []
for cluster in miner.drain.clusters:
    print(f"  [{cluster.cluster_id}] (count={cluster.size}): {cluster.get_template()}")
    template_data.append({
        'template_id': f"T_{cluster.cluster_id:03d}",  # Format ID dạng T_001, T_002 cho đồng bộ
        'template': cluster.get_template(),
        'count': cluster.size
    })

=== All Templates ===
  [1] (count=311): PacketResponder <*> for block <*> terminating
  [2] (count=314): BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
  [3] (count=292): Received block <*> of size <*> from <*>
  [4] (count=292): Receiving block <*> src: <*> dest: <*>
  [5] (count=115): BLOCK* NameSystem.allocateBlock: <*> <*>
  [6] (count=20): Verification succeeded for <*>
  [7] (count=263): Deleting block <*> file <*>
  [8] (count=80): <*> Served block <*> to <*>
  [9] (count=80): <*> exception while serving <*> to <*>
  [10] (count=224): BLOCK* NameSystem.delete: <*> is added to invalidSet of <*>
  [11] (count=1): 10.250.15.198:50010 Starting thread to transfer block blk_4292382298896622412 to 10.250.15.240:50010
  [12] (count=2): BLOCK* ask <*> to delete <*>
  [13] (count=2): Received block <*> src: <*> dest: <*> of size 67108864
  [14] (count=1): BLOCK* ask 10.250.17.177:50010 to delete blk_-8570780307468499817 blk_-9122557405432088649 blk_-4393

In [36]:
df_all_templates = pd.DataFrame(template_data)
df_all_templates = df_all_templates.sort_values(by='count', ascending=False).reset_index(drop=True)

print(f"Tổng số dòng log đã xử lý: {len(df_log_structured)} dòng")
print(f"Tổng số Templates duy nhất bóc tách được: {len(df_all_templates)}")

Tổng số dòng log đã xử lý: 2000 dòng
Tổng số Templates duy nhất bóc tách được: 17


In [37]:
print("\n=== BẢNG THỐNG KÊ TEMPLATES (SẮP XẾP THEO COUNT) ===")
print(df_all_templates[['template_id', 'count', 'template']])

path_top_10 = os.path.join('E:\WorkSpace\AIOPS\Week_01\w1\day-b', 'top_templates.csv')
df_top_10 = df_all_templates.head(10)

df_top_10.to_csv(path_top_10, index=False, encoding='utf-8')

print("\n" + "="*50)
print(f"Export Top-10 Templates ra: {path_top_10}")
print("="*50)
df_top_10


=== BẢNG THỐNG KÊ TEMPLATES (SẮP XẾP THEO COUNT) ===
   template_id  count                                           template
0        T_002    314  BLOCK* NameSystem.addStoredBlock: blockMap upd...
1        T_001    311      PacketResponder <*> for block <*> terminating
2        T_003    292            Received block <*> of size <*> from <*>
3        T_004    292             Receiving block <*> src: <*> dest: <*>
4        T_007    263                        Deleting block <*> file <*>
5        T_010    224  BLOCK* NameSystem.delete: <*> is added to inva...
6        T_005    115           BLOCK* NameSystem.allocateBlock: <*> <*>
7        T_009     80             <*> exception while serving <*> to <*>
8        T_008     80                        <*> Served block <*> to <*>
9        T_006     20                     Verification succeeded for <*>
10       T_012      2                       BLOCK* ask <*> to delete <*>
11       T_013      2  Received block <*> src: <*> dest: <*> of size .

,template_id,template,count
0,T_002,BLOCK* NameSystem.addStoredBlock: blockMap upd...,314
1,T_001,PacketResponder <*> for block <*> terminating,311
2,T_003,Received block <*> of size <*> from <*>,292
3,T_004,Receiving block <*> src: <*> dest: <*>,292
4,T_007,Deleting block <*> file <*>,263
5,T_010,BLOCK* NameSystem.delete: <*> is added to inva...,224
6,T_005,BLOCK* NameSystem.allocateBlock: <*> <*>,115
7,T_009,<*> exception while serving <*> to <*>,80
8,T_008,<*> Served block <*> to <*>,80
9,T_006,Verification succeeded for <*>,20


### Tune drain_sim_th

In [39]:
log_lines = df_log_structured['Content'].astype(str).tolist()
print(f"Processing {len(log_lines)} log lines...")
print("\n=== START TUNING DRAIN_SIM_TH ===")
sim_thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
best_sim_th = 0.5
best_miner = None
min_templates_count = float('inf') 

tuning_results = []
for th in sim_thresholds:
    config = TemplateMinerConfig()
    config.drain_sim_th = th
    config.drain_depth = 4 
    
    miner_tune = TemplateMiner(config=config)
    
    for line in log_lines:
        miner_tune.add_log_message(line)
        
    num_templates = len(miner_tune.drain.clusters)
    tuning_results.append({'sim_th': th, 'num_templates': num_templates})
    print(f"Với drain_sim_th = {th} | Sinh ra: {num_templates} templates unique")
    
    if th == 0.5:
        best_miner = miner_tune
        best_sim_th = th


Processing 2000 log lines...

=== START TUNING DRAIN_SIM_TH ===
Với drain_sim_th = 0.1 | Sinh ra: 16 templates unique
Với drain_sim_th = 0.3 | Sinh ra: 17 templates unique
Với drain_sim_th = 0.5 | Sinh ra: 17 templates unique
Với drain_sim_th = 0.7 | Sinh ra: 700 templates unique
Với drain_sim_th = 0.9 | Sinh ra: 1864 templates unique


Bảng kết quả Thực Nghiệm Tuning

| Kịch bản | Tham số `drain_sim_th` | Số lượng Templates Unique sinh ra | Đánh giá trạng thái hệ thống |
| :--- | :---: | :---: | :--- |
| **Lỏng lẻo** | `0.1` | 16 | **Under-parsing:** Mất mát thông tin (Gộp nhầm log khác loại). |
| **Baseline** | `0.3` | 17 | **Sweet Spot:** Cấu trúc hội tụ ổn định, bền vững. |
| **Đề xuất** | **0.5** | **17** | **Sweet Spot (Tối ưu):** Cân bằng hoàn hảo, lọc sạch biến số. |
| **Khắt khe** | `0.7` | 700 | **Over-parsing:** Bùng nổ template rác do chuỗi biến số động. |
| **Cực đoan** | `0.9` | 1864 | **Template Explosion:** Gần như mỗi dòng log biến thành 1 nhóm. |

Phân Tích & Biện Luận Hiện Tượng (Reflection)

Bản chất của thuật toán Drain3 hoạt động dựa trên cấu trúc phân nhánh cây (Parse Tree) phối hợp với việc đo độ tương đồng token. Kết quả thực nghiệm trên phản ánh rất rõ 3 phân vùng hành vi của mô hình:

1. **Hiện tượng hội tụ ở vùng thấp ($0.1 \le \text{sim\_th} \le 0.5$):**
   * Số lượng template được giữ ổn định ở mức **17**. Điều này chứng tỏ đặc tính dữ liệu log của hệ thống phân tán HDFS cực kỳ tường minh, các từ khóa tĩnh và tham số động có khoảng cách phân tách mạnh. 
   * Tuy nhiên, tại mức cực đoan `0.1`, hệ thống chỉ nhận diện được 16 cụm (bị hụt mất 1 cụm so với mức chuẩn), nghĩa là có 2 thông điệp mang ý nghĩa vận hành khác nhau đã bị thuật toán "dễ dãi" gộp chung lại làm một $\rightarrow$ *Gây mất mát dữ liệu cảnh báo*.

2. **Hiện tượng bùng nổ ở vùng cao ($0.7 \le \text{sim\_th} \le 0.9$):**
   * Khi đẩy `sim_th` lên cao, mô hình bắt buộc các dòng log phải giống nhau trên $70\%$ hoặc $90\%$ câu chữ thì mới gom chung cụm.
   * *Hậu quả:* Các dòng log mang lệnh xóa hoặc ghi nhận hàng loạt block (chứa chuỗi mã Hex hoặc mã Block dài như `blk_338095...` nối đuôi nhau) sẽ làm tỷ lệ tương đồng sụt giảm nghiêm trọng. Drain3 bị "bẫy" và coi mỗi dòng log chứa ID khác nhau là một loại lỗi hoàn toàn mới, dẫn đến bùng nổ **700 và 1864 template rác**.

Quyết định lựa chọn Production: `drain_sim_th = 0.5`

* **Lý do chọn:** Mức `0.5` là điểm **Sweet Spot** lý tưởng nhất. Nó vừa đảm bảo bóc tách sạch sẽ các biến số động như IP, Block ID, dung lượng file thành ký tự đại diện `<*>`, vừa giữ an toàn cho hệ thống không bị nổ cụm rác khi các chuỗi tham số dài biến động trong môi trường Production thực tế. 
* Bộ cấu hình này sẽ được đóng gói và chuyển tiếp sang Phase 2 để dựng chuỗi thời gian số học (`Template Count Time Series`).